In [1]:
import os
import shutil
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from PIL import Image

In [2]:
# -------------------------
# 1️⃣ Paths
# -------------------------
original_dataset = "/content/drive/MyDrive/MindScan/new_dataset"
binary_dataset = "/content/drive/MyDrive/MindScan/new_binary_dataset"

os.makedirs(binary_dataset, exist_ok=True)


In [3]:
tumor_classes = ["glioma", "meningioma", "pituitary"]
notumor_class = "notumor"
undersample_count = 667

In [4]:
# -------------------------
# 2️⃣ Create binary dataset
# -------------------------
# 2.1 Copy notumor
os.makedirs(os.path.join(binary_dataset, "notumor"), exist_ok=True)
notumor_src = os.path.join(original_dataset, notumor_class)
notumor_dst = os.path.join(binary_dataset, "notumor")
for fname in os.listdir(notumor_src):
    shutil.copy(os.path.join(notumor_src, fname), os.path.join(notumor_dst, fname))

# 2.2 Copy tumor images with undersampling
os.makedirs(os.path.join(binary_dataset, "tumor"), exist_ok=True)
for cls in tumor_classes:
    src_folder = os.path.join(original_dataset, cls)
    all_images = os.listdir(src_folder)
    sampled_images = random.sample(all_images, undersample_count)
    for fname in sampled_images:
        shutil.copy(os.path.join(src_folder, fname),
                    os.path.join(binary_dataset, "tumor", f"{cls}_{fname}"))


In [5]:
# -------------------------
# 3️⃣ Load image paths and labels
# -------------------------
image_paths = []
labels = []

for cls in ["notumor", "tumor"]:
    folder = os.path.join(binary_dataset, cls)
    for fname in os.listdir(folder):
        image_paths.append(os.path.join(folder, fname))
        labels.append(0 if cls == "notumor" else 1)  # 0=notumor, 1=tumor


In [6]:
# -------------------------
# 4️⃣ Split dataset reproducibly
# -------------------------
X_train, X_test_val, y_train, y_test_val = train_test_split(
    image_paths, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_test_val, y_test_val, test_size=0.5, random_state=42, stratify=y_test_val
)

print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")


Train size: 2800
Validation size: 600
Test size: 601


In [7]:
# -------------------------
# 5️⃣ PyTorch Dataset & DataLoader
# -------------------------
class CustomImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = CustomImageDataset(X_train, y_train, transform)
val_dataset = CustomImageDataset(X_val, y_val, transform)
test_dataset = CustomImageDataset(X_test, y_test, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [8]:
# -------------------------
# 6️⃣ Load MobileNetV2
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
# Replace classifier
model.classifier[1] = nn.Linear(model.last_channel, 2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 185MB/s]


In [9]:
# -------------------------
# 7️⃣ Training loop with validation
# -------------------------
num_epochs = 10
best_val_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for imgs, labels_batch in train_loader:
        imgs, labels_batch = imgs.to(device), labels_batch.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels_batch in val_loader:
            imgs, labels_batch = imgs.to(device), labels_batch.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)
    val_acc = correct / total

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_mobilenetv2.pth")

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
          f"Val Acc={val_acc:.4f}")


Epoch 1/10: Train Loss=0.2190, Train Acc=0.9396, Val Acc=0.9767
Epoch 2/10: Train Loss=0.0537, Train Acc=0.9846, Val Acc=0.9833
Epoch 3/10: Train Loss=0.0150, Train Acc=0.9982, Val Acc=0.9900
Epoch 4/10: Train Loss=0.0094, Train Acc=0.9993, Val Acc=0.9900
Epoch 5/10: Train Loss=0.0060, Train Acc=0.9989, Val Acc=0.9867
Epoch 6/10: Train Loss=0.0031, Train Acc=1.0000, Val Acc=0.9900
Epoch 7/10: Train Loss=0.0023, Train Acc=0.9996, Val Acc=0.9900
Epoch 8/10: Train Loss=0.0011, Train Acc=1.0000, Val Acc=0.9900
Epoch 9/10: Train Loss=0.0010, Train Acc=1.0000, Val Acc=0.9900
Epoch 10/10: Train Loss=0.0016, Train Acc=1.0000, Val Acc=0.9900


In [10]:
# -------------------------
# 8️⃣ Load best model and evaluate on test set
# -------------------------
model.load_state_dict(torch.load("best_mobilenetv2.pth"))
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for imgs, labels_batch in test_loader:
        imgs, labels_batch = imgs.to(device), labels_batch.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)
print(f"Test Accuracy: {correct/total:.4f}")


Test Accuracy: 0.9867


In [14]:
# ======= PREPROCESSING FUNCTIONS =======
import cv2
import numpy as np

def crop_black(img):
    """Crop black borders around the brain area."""
    coords = cv2.findNonZero(img)
    if coords is None:
        return img
    x, y, w, h = cv2.boundingRect(coords)
    return img[y:y+h, x:x+w]

def preprocess_image(img_path,  target_size=(224, 224)):
    """Full preprocessing pipeline for one image."""
    # 1. Read grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 2. Crop black borders
    img = crop_black(img)

    # 3. CLAHE contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)

    # 4. Bilateral denoising
    img = cv2.bilateralFilter(img, d=5, sigmaColor=20, sigmaSpace=50)

    # 5. Resize to target size
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)

    # 6. Normalize to [0, 1]
    img = img.astype(np.float32) / 255.0

    return img


In [16]:
# -------------------------
# 9️⃣ Test on a single image
# -------------------------

from PIL import Image
import torch

def predict_image(model, img_path):
    model.eval()
    img_np = preprocess_image(img_path)  # returns numpy array, shape (H,W)
    if img_np is None:
        return "Error reading image"

    # Convert grayscale NumPy array to PIL Image (mode "L"), then to RGB
    img = Image.fromarray((img_np * 255).astype(np.uint8)).convert("RGB")

    # Apply the same transform used in training
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img)
        _, pred = torch.max(output, 1)

    return "tumor" if pred.item() == 1 else "notumor"


# Example usage
img_path = "/content/meningioma_1.webp"
print("Predicted class:", predict_image(model, img_path))

Predicted class: tumor


In [19]:
# Example usage
img_path = "/content/meningioma_2.webp"
print("Predicted class:", predict_image(model, img_path))

Predicted class: tumor


In [18]:
# Example usage
img_path = "/content/normal_1.webp"
print("Predicted class:", predict_image(model, img_path))

Predicted class: notumor


In [20]:
# Example usage
img_path = "/content/pituitary_1.webp"
print("Predicted class:", predict_image(model, img_path))

Predicted class: tumor


In [21]:
# Example usage
img_path = "/content/normal_2.webp"
print("Predicted class:", predict_image(model, img_path))

Predicted class: notumor
